# Basic RAG Pipeline

A small, fully local retrieval-augmented generation (RAG) demo. Run the cells from top to bottom.

Pipeline: **create sample files → load → split and chunk → vectorize → retrieve → build a grounded answer**.

This notebook uses only the Python standard library, so it needs no API key, model download, or vector database.

## 1. Setup

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re

print("Ready. No external packages are required.")

## 2. Create and load sample documents

The first run creates three tiny `.txt` files in `rag_sample_docs/`. Later runs reuse them. Replace these files with your own text documents to test the same pipeline.

In [ ]:
data_dir = Path.cwd() / "rag_sample_docs"
data_dir.mkdir(exist_ok=True)

sample_documents = {
    "rag_overview.txt": (
        "Retrieval-augmented generation, or RAG, combines information retrieval with text generation. "
        "A RAG system first searches a document collection for passages related to a question. "
        "It then places the retrieved passages in the prompt so that a language model can produce a grounded answer."
    ),
    "chunking_notes.txt": (
        "Chunking divides a long document into smaller passages that can be searched independently. "
        "Chunk overlap repeats a small amount of text between neighboring chunks and helps preserve context near a boundary. "
        "Very small chunks may lose meaning, while very large chunks may contain too much unrelated information."
    ),
    "retrieval_notes.txt": (
        "Vector retrieval represents documents and questions as numeric vectors. "
        "Similar vectors are close in the vector space. Cosine similarity is a common way to rank them. "
        "The highest-scoring chunks are returned as context for the answer. Retrieval quality should be tested with realistic questions."
    ),
}

for filename, text in sample_documents.items():
    path = data_dir / filename
    if not path.exists():
        path.write_text(text, encoding="utf-8")

documents = [
    {"source": path.name, "text": path.read_text(encoding="utf-8")}
    for path in sorted(data_dir.glob("*.txt"))
]

print(f"Loaded {len(documents)} documents from {data_dir}")
for document in documents:
    print(f"- {document['source']}: {len(document['text'])} characters")

## 3. Clean, split, and chunk

First normalize whitespace and split each document into sentences. Then build overlapping word chunks. In a production pipeline, chunk size is normally measured in model tokens rather than words.

In [ ]:
def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def split_into_sentences(text):
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", clean_text(text)) if part.strip()]

sentences = []
for document in documents:
    for sentence_id, sentence in enumerate(split_into_sentences(document["text"])):
        sentences.append({
            "source": document["source"],
            "sentence_id": sentence_id,
            "text": sentence,
        })

print(f"Split the documents into {len(sentences)} sentences.")
sentences[:3]

In [ ]:
def chunk_text(text, chunk_size=40, overlap=10):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = clean_text(text).split()
    step = chunk_size - overlap
    return [" ".join(words[start:start + chunk_size]) for start in range(0, len(words), step)]

chunks = []
for document in documents:
    for chunk_id, text in enumerate(chunk_text(document["text"], chunk_size=40, overlap=10)):
        chunks.append({
            "source": document["source"],
            "chunk_id": chunk_id,
            "text": text,
        })

print(f"Created {len(chunks)} overlapping chunks.")
for chunk in chunks:
    print(f"[{chunk['source']} / chunk {chunk['chunk_id']}] {chunk['text']}\n")

## 4. Vectorize chunks with TF-IDF

TF-IDF gives higher weights to words that are important in one chunk but less common across the collection. Each chunk becomes a sparse numeric vector.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_chunks = [tokenize(chunk["text"]) for chunk in chunks]
document_frequency = Counter()
for tokens in tokenized_chunks:
    document_frequency.update(set(tokens))

number_of_chunks = len(chunks)
idf = {
    term: math.log((1 + number_of_chunks) / (1 + frequency)) + 1
    for term, frequency in document_frequency.items()
}

def tfidf_vector(text):
    tokens = tokenize(text)
    if not tokens:
        return {}
    counts = Counter(tokens)
    return {term: (count / len(tokens)) * idf.get(term, 0.0) for term, count in counts.items()}

chunk_vectors = [tfidf_vector(chunk["text"]) for chunk in chunks]
print(f"Vectorized {len(chunk_vectors)} chunks with a vocabulary of {len(idf)} terms.")
print("Example vector (first 10 entries):", list(chunk_vectors[0].items())[:10])

## 5. Query and retrieve the most similar chunks

The query is vectorized with the same TF-IDF vocabulary. Cosine similarity ranks chunks by the angle between the query and chunk vectors.

In [ ]:
def cosine_similarity(vector_a, vector_b):
    shared_terms = set(vector_a) & set(vector_b)
    dot_product = sum(vector_a[term] * vector_b[term] for term in shared_terms)
    norm_a = math.sqrt(sum(value ** 2 for value in vector_a.values()))
    norm_b = math.sqrt(sum(value ** 2 for value in vector_b.values()))
    return dot_product / (norm_a * norm_b) if norm_a and norm_b else 0.0

def retrieve(query, top_k=3):
    query_vector = tfidf_vector(query)
    ranked = []
    for chunk, vector in zip(chunks, chunk_vectors):
        ranked.append({**chunk, "score": cosine_similarity(query_vector, vector)})
    return sorted(ranked, key=lambda item: item["score"], reverse=True)[:top_k]

query = "Why is overlap useful when splitting documents into chunks?"
results = retrieve(query, top_k=3)

print("Query:", query)
for rank, result in enumerate(results, start=1):
    print(f"\n{rank}. score={result['score']:.3f} | {result['source']} | chunk {result['chunk_id']}")
    print(result["text"])

## 6. Build the augmented prompt and a simple answer

In a full RAG system, the prompt below would be sent to an LLM. To keep this notebook offline, the demo answer selects the most relevant sentence from the retrieved context.

In [ ]:
context = "\n\n".join(
    f"Source: {result['source']}\n{result['text']}"
    for result in results
)

augmented_prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say that the available documents do not provide enough information.
Cite the source filename.

Context:
{context}

Question: {query}
Answer:"""

print(augmented_prompt)

In [ ]:
def extractive_answer(query, retrieved_chunks):
    query_terms = set(tokenize(query))
    candidates = []
    for result in retrieved_chunks:
        for sentence in split_into_sentences(result["text"]):
            overlap_score = len(query_terms & set(tokenize(sentence)))
            candidates.append((overlap_score, result["score"], sentence, result["source"]))
    best = max(candidates, key=lambda item: (item[0], item[1]))
    return f"{best[2]} (Source: {best[3]})"

answer = extractive_answer(query, results)
print("Demo answer:", answer)

## 7. Try another question

Edit `my_query` and rerun this cell.

In [ ]:
my_query = "Where are retrieved passages placed before answer generation?"
my_results = retrieve(my_query, top_k=2)

print("Question:", my_query)
print("Answer:", extractive_answer(my_query, my_results))
print("\nRetrieved evidence:")
for result in my_results:
    print(f"- {result['source']} (score={result['score']:.3f}): {result['text']}")

## What this demo leaves out

A production RAG application would usually add PDF/Word loaders, model-based embeddings, a persistent vector database, an LLM generation call, metadata filters, citations, and retrieval evaluation. The data flow in this notebook stays the same even when those components are replaced.